In [1]:
import os
import json
from pathlib import Path

import pandas as pd
import textstat

from google import genai

print("Baseline notebook setup successful!")

Baseline notebook setup successful!


In [2]:
import sys
print(sys.executable)

/opt/anaconda3/envs/google2e/bin/python


In [3]:
import sys
import pandas as pd
import textstat
from google import genai

print("Python:", sys.version)
print("Environment:", sys.executable)
print("All imports successful!")

Python: 3.15.0 | packaged by conda-forge | (main, Oct 12 2026, 17:28:19) [Clang 21.1.8 ]
Environment: /opt/anaconda3/envs/google2e/bin/python
All imports successful!


In [4]:
# A small example to test our single-Gemini baseline

sample_medical_text = """
The patient presents with hypertension and elevated blood glucose.
Lifestyle modifications, including dietary changes and regular
physical activity, are recommended to reduce cardiovascular risk.
"""

print(sample_medical_text)


The patient presents with hypertension and elevated blood glucose.
Lifestyle modifications, including dietary changes and regular
physical activity, are recommended to reduce cardiovascular risk.



In [5]:
# Fixed instructions for our single-Gemini baseline

BASELINE_PROMPT = """
You are a medical information simplification assistant.

Rewrite the provided medical text as a clear, patient-friendly explanation.

Follow these rules:
1. Use plain, everyday English.
2. Aim for an 8th-grade reading level or below.
3. Explain medical terminology using simple language.
4. Preserve all clinically important information from the original text.
5. Do not invent symptoms, diagnoses, treatments, or medical facts.
6. Do not make recommendations that are not supported by the source text.
7. Do not remove important warnings, measurements, or uncertainty.
8. Return only the rewritten explanation.

Medical text:
{medical_text}
"""

# Insert our first medical example into the prompt
test_prompt = BASELINE_PROMPT.format(
    medical_text=sample_medical_text
)

print(test_prompt)


You are a medical information simplification assistant.

Rewrite the provided medical text as a clear, patient-friendly explanation.

Follow these rules:
1. Use plain, everyday English.
2. Aim for an 8th-grade reading level or below.
3. Explain medical terminology using simple language.
4. Preserve all clinically important information from the original text.
5. Do not invent symptoms, diagnoses, treatments, or medical facts.
6. Do not make recommendations that are not supported by the source text.
7. Do not remove important warnings, measurements, or uncertainty.
8. Return only the rewritten explanation.

Medical text:

The patient presents with hypertension and elevated blood glucose.
Lifestyle modifications, including dietary changes and regular
physical activity, are recommended to reduce cardiovascular risk.




In [6]:
from getpass import getpass
import os
from google import genai

# Ask for the API key without displaying it on screen
os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

# Create the Gemini client
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("Gemini client initialized!")


Gemini client initialized!


In [7]:
import time
from google.genai import errors

MODEL_NAME = "gemini-3.5-flash-lite"

baseline_output = None

for attempt in range(3):
    try:
        print(f"Attempt {attempt + 1}...")

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=test_prompt,
        )

        baseline_output = response.text
        break

    except errors.ServerError as e:
        if e.code == 503 and attempt < 2:
            print("Gemini is busy. Waiting 15 seconds...")
            time.sleep(15)
        else:
            raise

if baseline_output:
    print("\nORIGINAL MEDICAL TEXT:")
    print(sample_medical_text)

    print("\nGEMINI'S PATIENT-FRIENDLY EXPLANATION:")
    print(baseline_output)

Attempt 1...

ORIGINAL MEDICAL TEXT:

The patient presents with hypertension and elevated blood glucose.
Lifestyle modifications, including dietary changes and regular
physical activity, are recommended to reduce cardiovascular risk.


GEMINI'S PATIENT-FRIENDLY EXPLANATION:
You have high blood pressure and high blood sugar. To lower your chances of heart and blood vessel problems, you are advised to make lifestyle changes, which include changing your diet and getting regular exercise.


In [8]:
from eval_harness import readability_scores, meets_readability_target

scores = readability_scores(baseline_output)

print("Gemini output:")
print(baseline_output)

print("\nReadability scores:")
print(scores)

print("\nMeets 8th-grade target:")
print(meets_readability_target(baseline_output))

Gemini output:
You have high blood pressure and high blood sugar. To lower your chances of heart and blood vessel problems, you are advised to make lifestyle changes, which include changing your diet and getting regular exercise.

Readability scores:
{'flesch_kincaid_grade': 8.766428571428573, 'smog_index': 8.841846274778883, 'jargon_density': 0.34285714285714286, 'word_count': 35}

Meets 8th-grade target:
False


In [9]:
original_scores = readability_scores(sample_medical_text)

print("ORIGINAL TEXT SCORES:")
print(original_scores)

print("\nGEMINI OUTPUT SCORES:")
print(scores)

print(
    "\nGrade-level improvement:",
    round(
        original_scores["flesch_kincaid_grade"]
        - scores["flesch_kincaid_grade"],
        2
    )
)

ORIGINAL TEXT SCORES:
{'flesch_kincaid_grade': 18.59, 'smog_index': 15.903189008614273, 'jargon_density': 0.6666666666666666, 'word_count': 24}

GEMINI OUTPUT SCORES:
{'flesch_kincaid_grade': 8.766428571428573, 'smog_index': 8.841846274778883, 'jargon_density': 0.34285714285714286, 'word_count': 35}

Grade-level improvement: 9.82


In [10]:
result = {
    "example_id": "test_001",
    "source_text": sample_medical_text.strip(),
    "gemini_output": baseline_output.strip(),
    "model": MODEL_NAME,
    "source_grade": original_scores["flesch_kincaid_grade"],
    "output_grade": scores["flesch_kincaid_grade"],
    "grade_improvement": (
        original_scores["flesch_kincaid_grade"]
        - scores["flesch_kincaid_grade"]
    ),
    "smog_index": scores["smog_index"],
    "jargon_density": scores["jargon_density"],
    "word_count": scores["word_count"],
    "meets_grade_8": meets_readability_target(baseline_output),
}

result

{'example_id': 'test_001',
 'source_text': 'The patient presents with hypertension and elevated blood glucose.\nLifestyle modifications, including dietary changes and regular\nphysical activity, are recommended to reduce cardiovascular risk.',
 'gemini_output': 'You have high blood pressure and high blood sugar. To lower your chances of heart and blood vessel problems, you are advised to make lifestyle changes, which include changing your diet and getting regular exercise.',
 'model': 'gemini-3.5-flash-lite',
 'source_grade': 18.59,
 'output_grade': 8.766428571428573,
 'grade_improvement': 9.823571428571427,
 'smog_index': 8.841846274778883,
 'jargon_density': 0.34285714285714286,
 'word_count': 35,
 'meets_grade_8': False}

In [11]:
import re
import time
from google.genai import errors

# Free tier allows 15 requests/minute for this model, so space calls ~4s apart
MIN_SECONDS_BETWEEN_CALLS = 60 / 15 + 0.5
MAX_ATTEMPTS = 5
_last_call_time = 0.0


def _retry_delay_seconds(error, default=15):
    # Gemini includes a suggested wait like "Please retry in 4.23s." in 429 errors
    match = re.search(r"retry in ([\d.]+)s", str(error))
    return float(match.group(1)) + 1 if match else default


def generate_with_retry(prompt):
    global _last_call_time

    for attempt in range(MAX_ATTEMPTS):
        # Throttle so we stay under the per-minute quota
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - _last_call_time)
        if wait > 0:
            time.sleep(wait)
        _last_call_time = time.time()

        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
            )

        except errors.ClientError as e:
            # 429 = rate limit hit; wait the suggested time and retry
            if e.code == 429 and attempt < MAX_ATTEMPTS - 1:
                delay = _retry_delay_seconds(e)
                print(
                    f"Rate limited. Retrying in {delay:.0f} seconds... "
                    f"(attempt {attempt + 1}/{MAX_ATTEMPTS})"
                )
                time.sleep(delay)
            else:
                raise

        except errors.ServerError as e:
            # 503 = Gemini temporarily busy
            if e.code == 503 and attempt < MAX_ATTEMPTS - 1:
                print(
                    f"Gemini busy. Retrying in 15 seconds... "
                    f"(attempt {attempt + 1}/{MAX_ATTEMPTS})"
                )
                time.sleep(15)
            else:
                raise


def run_baseline(example_id, medical_text):
    prompt = BASELINE_PROMPT.format(
        medical_text=medical_text
    )

    response = generate_with_retry(prompt)

    output = response.text.strip()

    source_scores = readability_scores(medical_text)
    output_scores = readability_scores(output)

    return {
        "example_id": example_id,
        "source_text": medical_text.strip(),
        "gemini_output": output,
        "model": MODEL_NAME,
        "source_grade": source_scores["flesch_kincaid_grade"],
        "output_grade": output_scores["flesch_kincaid_grade"],
        "grade_improvement": (
            source_scores["flesch_kincaid_grade"]
            - output_scores["flesch_kincaid_grade"]
        ),
        "smog_index": output_scores["smog_index"],
        "jargon_density": output_scores["jargon_density"],
        "word_count": output_scores["word_count"],
        "meets_grade_8": meets_readability_target(output),
    }

print("Baseline function with rate limiting and retry protection created!")

Baseline function with rate limiting and retry protection created!


In [12]:
test_result = run_baseline(
    example_id="test_002",
    medical_text=sample_medical_text
)

test_result

{'example_id': 'test_002',
 'source_text': 'The patient presents with hypertension and elevated blood glucose.\nLifestyle modifications, including dietary changes and regular\nphysical activity, are recommended to reduce cardiovascular risk.',
 'gemini_output': 'You have high blood pressure and high blood sugar. To lower your chances of heart and blood vessel problems, you are advised to change your lifestyle by eating a healthier diet and getting regular exercise.',
 'model': 'gemini-3.5-flash-lite',
 'source_grade': 18.59,
 'output_grade': 8.766428571428573,
 'grade_improvement': 9.823571428571427,
 'smog_index': 10.125756701596842,
 'jargon_density': 0.3142857142857143,
 'word_count': 35,
 'meets_grade_8': False}

In [13]:
from pathlib import Path

data_dir = Path("../data")

print("Files available in data/:")
for file in sorted(data_dir.iterdir()):
    print("-", file.name)

Files available in data/:
- .DS_Store
- .gitkeep
- MedAESQA_methods_M1-M30.xlsx
- README.md
- baseline_50_full.csv
- baseline_50_ids.csv
- medaesqa_v1.json
- mtsamples.csv
- retrieval_docs_medlineplus_chunked.jsonl


In [14]:
from pathlib import Path

project_root = Path("..")

search_terms = [
    "baseline",
    "50",
    "PLABA",
    "MTSamples",
    "gold set",
    "frozen"
]

for path in project_root.rglob("*"):
    if path.is_file() and path.suffix in {".py", ".md"}:
        try:
            text = path.read_text(errors="ignore")
            matches = [
                term for term in search_terms
                if term.lower() in text.lower()
            ]

            if matches:
                print(f"{path}: {matches}")
        except Exception:
            pass

../README.md: ['baseline']
../SUCCESS_RUBRIC.md: ['baseline', '50']
../Challenge-Project-Overview.md: ['baseline', '50', 'PLABA', 'MTSamples', 'gold set', 'frozen']
../data/README.md: ['PLABA', 'MTSamples', 'gold set', 'frozen']


In [15]:
# Run the baseline on every entry in a CSV file

INPUT_CSV = Path("../data/baseline_50_full.csv")
OUTPUT_CSV = Path("../data/baseline_50_gemini_outputs.csv")
ID_COLUMN = "row_id"
TEXT_COLUMN = "text"

input_df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(input_df)} entries from {INPUT_CSV}")

# Resume from a previous run so already-finished rows aren't re-sent
results = []
if OUTPUT_CSV.exists():
    results = pd.read_csv(OUTPUT_CSV).to_dict("records")
done_ids = {str(r["example_id"]) for r in results}
print(f"{len(done_ids)} entries already done; skipping those")

failed_ids = []

for i, row in input_df.iterrows():
    if str(row[ID_COLUMN]) in done_ids:
        continue

    print(f"[{i + 1}/{len(input_df)}] {row[ID_COLUMN]}")

    try:
        results.append(
            run_baseline(
                example_id=row[ID_COLUMN],
                medical_text=row[TEXT_COLUMN]
            )
        )
        # Save after every row so progress survives a crash or quota cutoff
        pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    except Exception as e:
        print(f"Failed on {row[ID_COLUMN]}: {e}")
        failed_ids.append(row[ID_COLUMN])

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved {len(results_df)} results to {OUTPUT_CSV}")
if failed_ids:
    print(f"{len(failed_ids)} failed (re-run this cell to retry): {failed_ids}")
results_df.head()

Loaded 50 entries from ../data/baseline_50_full.csv
0 entries already done; skipping those
[1/50] 1896
[2/50] 3527
[3/50] 1910
[4/50] 1916
[5/50] 39
[6/50] 540
[7/50] 620
[8/50] 1656
[9/50] 1730
[10/50] 1355
[11/50] 1285
[12/50] 1810
[13/50] 2769
[14/50] 2807
[15/50] 3220
[16/50] 3278
[17/50] 3291
[18/50] 132
[19/50] 1917
[20/50] 3404
[21/50] 3414
[22/50] 3419
[23/50] 3539
[24/50] 3662
[25/50] 3942
[26/50] 2214
[27/50] 2575
[28/50] 2602
[29/50] 3121
[30/50] 3274
[31/50] 167
[32/50] 466
[33/50] 715
[34/50] 742
[35/50] 751
[36/50] 819
[37/50] 858
[38/50] 1583
[39/50] 1652
[40/50] 1658
[41/50] 1717
[42/50] 1732
[43/50] 1750
[44/50] 34
[45/50] 60
[46/50] 121
[47/50] 133
[48/50] 1122
[49/50] 1325
[50/50] 1336

Saved 50 results to ../data/baseline_50_gemini_outputs.csv


,example_id,source_text,gemini_output,model,source_grade,output_grade,grade_improvement,smog_index,jargon_density,word_count,meets_grade_8
0,1896,"HISTORY OF PRESENT ILLNESS:, This is a 1-year-...",This is the medical report for a 1-year-old bo...,gemini-3.5-flash-lite,8.074680,7.222498,0.852183,10.614719,0.224880,627,True
1,3527,"PROBLEM: ,Rectal bleeding, positive celiac spr...",**Patient Information**\n\n* **Age and backgro...,gemini-3.5-flash-lite,10.297203,8.475714,1.821488,10.371328,0.297619,336,False
2,1910,"ADMITTING DIAGNOSES:,1. Fever.,2. Otitis media...",**Why the Patient Was Admitted**\nA 10-month-o...,gemini-3.5-flash-lite,8.360764,8.239715,0.121049,10.980519,0.184258,559,False
3,1916,"ADMITTING DIAGNOSES:, Respiratory distress syn...",**Main Health Conditions at Admission**\n* *...,gemini-3.5-flash-lite,10.281765,9.407518,0.874246,11.476623,0.193182,616,False
4,39,"CHIEF COMPLAINT: , Testicular pain.,HISTORY OF...",**Main Reason for Visit:**\nA 4-year-old boy c...,gemini-3.5-flash-lite,9.267349,6.894536,2.372813,10.393551,0.182274,598,True
